# Tiền xử lý `price` và `area` cho dataset Phongtro123 – TP.HCM  

Notebook này tập trung **tiền xử lý 2 trường quan trọng** cho mô hình:  

- `price`  → chuyển về **giá tiền VND/tháng (kiểu số)**  
- `area`   → chuyển về **diện tích m² (kiểu số)**  

Đồng thời chuẩn hóa lại schema để đồng bộ với các dataset khác:  

> `url (str), title (str), price (number), area (number), address (str), description (str), source (str)`  

Giả định file dữ liệu thô được lưu tại:  
`phongtro123_raw.csv`  
(do crawler xuất ra trước đó).


## 1. Import thư viện & đọc dữ liệu thô  

Ở bước này ta:  

- Import các thư viện cần thiết (`pandas`, `numpy`, `re`).  
- Đọc file `phongtro123_raw.csv`.  
- Xem sơ qua vài dòng đầu và danh sách cột để kiểm tra cấu trúc.


In [1]:
import pandas as pd
import numpy as np
import re

# Đường dẫn tới file CSV thô do crawler sinh ra
raw_path = "phongtro123_raw.csv"

df = pd.read_csv(raw_path)

print("Số dòng, số cột:", df.shape)
print("\nDanh sách cột:")
print(df.columns.tolist())

df.head()

Số dòng, số cột: (12000, 10)

Danh sách cột:
['url', 'title', 'price', 'area', 'address', 'description', 'posted_time', 'owner_name', 'phone', 'source']


,url,title,price,area,address,description,posted_time,owner_name,phone,source
0,https://phongtro123.com/chung-cu-thai-an-nguye...,"Chung cư Thái An, Nguyễn Văn Quá, Q12: 110m2, ...",9.5 triệu/tháng,110 m2,"Đường Nguyễn Văn Quá, Phường Đông Hưng Thuận, ...","Chung cư Thái An, Nguyễn Văn Quá, Q12: 110m2, ...",10:49 30/11/2025,Tú Oanh,905135147,phongtro123
1,https://phongtro123.com/chinh-chu-cho-thue-chd...,Chính chủ cho thuê CHDV mới xây tại VẠN PHÚC C...,7 triệu/tháng,40 m2,"Thủ Đức, Hồ Chí Minh","Khai trương căn hộ mới xây, trong khu đô thị V...",11:51 30/11/2025,Hồng Hiên,964602883,phongtro123
2,https://phongtro123.com/phong-gac-cua-so-thoan...,"Phòng gác cửa sổ thoáng, 2 chỗ ngủ giá rẻ gần ...",4.9 triệu/tháng,30 m2,"458 Đường Huỳnh Tấn Phát, Phường Bình Thuận, Q...","PHÒNG GÁC 2 CHỖ NGỦ GIÁ RẺ, FULL NỘI THẤT CẠNH...",10:41 30/11/2025,Minh Luân Megas,939031545,phongtro123
3,https://phongtro123.com/ktx-cho-thue-theo-tuan...,KTX CHO THUÊ THEO TUẦN – Ở NGAY – CHỈ TỪ 600K/...,1.75 triệu/tháng,20 m2,"33a Đường Ngô Quyền, Phường Hiệp Phú, Quận 9, ...",Bạn cần ở tạm 1 tuần – 2 tuần – 3 tuần – 30 ng...,08:04 29/11/2025,Lê Thế Anh,938120264,phongtro123
4,https://phongtro123.com/chdv-full-noi-that-ban...,"Phòng full nội thất, ban công - cửa sổ, mặt ti...",5 triệu/tháng,25 m2,"Đường Phan Đình Phùng, Phường 2, Quận Phú Nhuậ...","Phòng full nội thất, ban công - cửa sổ, mặt ti...",20:34 21/11/2025,Trần Phạm Minh Tiến,928161541,phongtro123


## 2. Chuẩn hóa schema: drop 3 cột không dùng  

Để đồng bộ schema với các nguồn dữ liệu khác, ta **bỏ 3 cột**:  

- `posted_time` – chỉ dùng cho phân tích theo thời gian, không dùng trong mô hình giá hiện tại.  
- `owner_name`  – thông tin chủ tin, dễ gây nhiễu.  
- `phone`       – thông tin nhạy cảm, không cần cho mô hình.  

Ta giữ lại những trường quan trọng hơn cho bài toán mô hình giá.


In [2]:
cols_to_drop = ["posted_time", "owner_name", "phone"]

# Chỉ drop những cột thực sự tồn tại (tránh lỗi nếu tên cột khác nhau)
cols_to_drop_existing = [c for c in cols_to_drop if c in df.columns]

df_proc = df.drop(columns=cols_to_drop_existing)

print("Còn lại các cột:")
print(df_proc.columns.tolist())

df_proc.head()

Còn lại các cột:
['url', 'title', 'price', 'area', 'address', 'description', 'source']


,url,title,price,area,address,description,source
0,https://phongtro123.com/chung-cu-thai-an-nguye...,"Chung cư Thái An, Nguyễn Văn Quá, Q12: 110m2, ...",9.5 triệu/tháng,110 m2,"Đường Nguyễn Văn Quá, Phường Đông Hưng Thuận, ...","Chung cư Thái An, Nguyễn Văn Quá, Q12: 110m2, ...",phongtro123
1,https://phongtro123.com/chinh-chu-cho-thue-chd...,Chính chủ cho thuê CHDV mới xây tại VẠN PHÚC C...,7 triệu/tháng,40 m2,"Thủ Đức, Hồ Chí Minh","Khai trương căn hộ mới xây, trong khu đô thị V...",phongtro123
2,https://phongtro123.com/phong-gac-cua-so-thoan...,"Phòng gác cửa sổ thoáng, 2 chỗ ngủ giá rẻ gần ...",4.9 triệu/tháng,30 m2,"458 Đường Huỳnh Tấn Phát, Phường Bình Thuận, Q...","PHÒNG GÁC 2 CHỖ NGỦ GIÁ RẺ, FULL NỘI THẤT CẠNH...",phongtro123
3,https://phongtro123.com/ktx-cho-thue-theo-tuan...,KTX CHO THUÊ THEO TUẦN – Ở NGAY – CHỈ TỪ 600K/...,1.75 triệu/tháng,20 m2,"33a Đường Ngô Quyền, Phường Hiệp Phú, Quận 9, ...",Bạn cần ở tạm 1 tuần – 2 tuần – 3 tuần – 30 ng...,phongtro123
4,https://phongtro123.com/chdv-full-noi-that-ban...,"Phòng full nội thất, ban công - cửa sổ, mặt ti...",5 triệu/tháng,25 m2,"Đường Phan Đình Phùng, Phường 2, Quận Phú Nhuậ...","Phòng full nội thất, ban công - cửa sổ, mặt ti...",phongtro123


## 3. Tiền xử lý `price` → số VND/tháng  

**Giả định trong dataset này chỉ có 2 dạng:**  

1. `... triệu/tháng`  
   - Ví dụ: `4.9 triệu/tháng`, `1.75 triệu/tháng`  
   - Dấu chấm trong phần số là **dấu thập phân**.  
   - Công thức: `value_million * 1_000_000`  

2. `... đồng/tháng` (hoặc `đ/tháng`, `vnđ/tháng`, `vnd/tháng`...)  
   - Ví dụ: `3.500.000 đ/tháng`  
   - Dấu chấm là **phân cách hàng nghìn**, không phải thập phân.  
   - Xử lý: xóa hết `.`/`,` rồi cast sang `int`.  

Mục tiêu: tạo cột mới `price` dạng **int (VND/tháng)**.


In [3]:
def parse_price_to_vnd(x: str):
    """
    Chuẩn hóa cột price:
    - 'x.y triệu/tháng' -> x.y * 1_000_000 (VND)
    - 'x.yyy.zzz đ/tháng' -> remove '.', ',' -> int
    - Trường hợp khác -> NaN
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if not s:
        return np.nan

    # Lấy phần số đầu tiên trong chuỗi (vd: '4.9', '3.500.000')
    m = re.search(r"[\d\.,]+", s)
    if not m:
        return np.nan
    num_str = m.group(0)

    # Case 1: triệu/tháng -> dấu chấm/phẩy là thập phân
    if "triệu" in s:
        # Ví dụ có thể gặp: '4.9' hoặc '4,9'
        num_str_norm = num_str.replace(",", ".")
        try:
            value_million = float(num_str_norm)
            return int(round(value_million * 1_000_000))
        except ValueError:
            return np.nan

    # Case 2: đồng/tháng -> dấu chấm là phân cách nghìn
    if any(k in s for k in ["đ", "dong", "đồng", "vnđ", "vnd"]):
        # Bỏ hết ký tự không phải số
        digits = re.sub(r"[^\d]", "", num_str)
        if digits == "":
            return np.nan
        try:
            return int(digits)
        except ValueError:
            return np.nan

    # Các trường hợp khác (hiếm) có thể xử lý riêng sau
    return np.nan

# Áp dụng vào cột price gốc
df_proc["price"] = df_proc["price"].apply(parse_price_to_vnd)

print("Một vài giá trị price trước và sau khi chuẩn hóa:")
print(pd.DataFrame({
    "price_raw": df["price"].head(10),
    "price_parsed": df_proc["price"].head(10)
}))

print("\nSố lượng price bị NaN:", df_proc["price"].isna().sum())

Một vài giá trị price trước và sau khi chuẩn hóa:
          price_raw  price_parsed
0   9.5 triệu/tháng       9500000
1     7 triệu/tháng       7000000
2   4.9 triệu/tháng       4900000
3  1.75 triệu/tháng       1750000
4     5 triệu/tháng       5000000
5   5.8 triệu/tháng       5800000
6   5.5 triệu/tháng       5500000
7  10.5 triệu/tháng      10500000
8     5 triệu/tháng       5000000
9    11 triệu/tháng      11000000

Số lượng price bị NaN: 0


## 4. Tiền xử lý `area` → số m² (float)  

Cột `area` thường có dạng:  

- `25 m2`, `40 m2`, `16 m²`, thi thoảng có dạng `25,5 m2`.  

Chiến lược:  

- Lấy phần số đầu tiên.  
- Xem **`.` hoặc `,` là dấu thập phân** (vì diện tích hiếm khi có tới hàng nghìn).  
- Chuẩn hóa về float (m²).  


In [4]:
def parse_area_to_m2(x: str):
    """
    Chuẩn hóa diện tích về số m2 (float).
    Ví dụ: '25 m2', '16 m²', '25,5 m2' -> 25.0, 16.0, 25.5
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if not s:
        return np.nan

    # Lấy phần số đầu tiên
    m = re.search(r"[\d\.,]+", s)
    if not m:
        return np.nan
    num_str = m.group(0)

    # Nếu có cả . và , thì xử lý bảo thủ: bỏ ., thay , bằng .
    if "." in num_str and "," in num_str:
        num_str_norm = num_str.replace(".", "").replace(",", ".")
    else:
        num_str_norm = num_str.replace(",", ".")

    try:
        return float(num_str_norm)
    except ValueError:
        return np.nan

df_proc["area"] = df_proc["area"].apply(parse_area_to_m2)

print("Một vài giá trị area trước và sau khi chuẩn hóa:")
print(pd.DataFrame({
    "area_raw": df["area"].head(10),
    "area_parsed": df_proc["area"].head(10)
}))

print("\nSố lượng area bị NaN:", df_proc["area"].isna().sum())

Một vài giá trị area trước và sau khi chuẩn hóa:
  area_raw  area_parsed
0   110 m2        110.0
1    40 m2         40.0
2    30 m2         30.0
3    20 m2         20.0
4    25 m2         25.0
5    32 m2         32.0
6    30 m2         30.0
7    50 m2         50.0
8    35 m2         35.0
9    50 m2         50.0

Số lượng area bị NaN: 0


## 5. Sắp xếp lại cột theo schema chuẩn  

Sau khi tiền xử lý, ta giữ lại đúng các cột cần thiết và sắp xếp theo thứ tự:  

> `url, title, price, area, address, description, source`  

Đây sẽ là format chuẩn để merge với các nguồn dữ liệu bất động sản khác.


In [5]:
expected_cols = ["url", "title", "price", "area", "address", "description", "source"]

# Chỉ giữ lại những cột có trong df_proc
cols_final = [c for c in expected_cols if c in df_proc.columns]

df_final = df_proc[cols_final].copy()

print("Schema cuối cùng:")
print(df_final.dtypes)
df_final.head()

Schema cuối cùng:
url             object
title           object
price            int64
area           float64
address         object
description     object
source          object
dtype: object


,url,title,price,area,address,description,source
0,https://phongtro123.com/chung-cu-thai-an-nguye...,"Chung cư Thái An, Nguyễn Văn Quá, Q12: 110m2, ...",9500000,110.0,"Đường Nguyễn Văn Quá, Phường Đông Hưng Thuận, ...","Chung cư Thái An, Nguyễn Văn Quá, Q12: 110m2, ...",phongtro123
1,https://phongtro123.com/chinh-chu-cho-thue-chd...,Chính chủ cho thuê CHDV mới xây tại VẠN PHÚC C...,7000000,40.0,"Thủ Đức, Hồ Chí Minh","Khai trương căn hộ mới xây, trong khu đô thị V...",phongtro123
2,https://phongtro123.com/phong-gac-cua-so-thoan...,"Phòng gác cửa sổ thoáng, 2 chỗ ngủ giá rẻ gần ...",4900000,30.0,"458 Đường Huỳnh Tấn Phát, Phường Bình Thuận, Q...","PHÒNG GÁC 2 CHỖ NGỦ GIÁ RẺ, FULL NỘI THẤT CẠNH...",phongtro123
3,https://phongtro123.com/ktx-cho-thue-theo-tuan...,KTX CHO THUÊ THEO TUẦN – Ở NGAY – CHỈ TỪ 600K/...,1750000,20.0,"33a Đường Ngô Quyền, Phường Hiệp Phú, Quận 9, ...",Bạn cần ở tạm 1 tuần – 2 tuần – 3 tuần – 30 ng...,phongtro123
4,https://phongtro123.com/chdv-full-noi-that-ban...,"Phòng full nội thất, ban công - cửa sổ, mặt ti...",5000000,25.0,"Đường Phan Đình Phùng, Phường 2, Quận Phú Nhuậ...","Phòng full nội thất, ban công - cửa sổ, mặt ti...",phongtro123


## 6. (Tuỳ chọn) Lưu dataset đã chuẩn hóa  

Nếu cần dùng cho các notebook EDA / modeling khác, ta có thể lưu lại thành file mới:  


In [6]:
clean_path = "phongtro123_clean_price_area.csv"
df_final.to_csv(clean_path, index=False, encoding="utf-8-sig")
print("Đã lưu file cleaned tại:", clean_path)

Đã lưu file cleaned tại: phongtro123_clean_price_area.csv
